In [2]:
import pandas as pd
import ast
from collections import defaultdict, Counter

In [24]:
res_1 = pd.read_json('annotator1.jsonl', lines=True)
res_2 = pd.read_json('annotator2.jsonl', lines=True)
res_3 = pd.read_json('annotator3.jsonl', lines=True)

In [ ]:
def iou(span1, span2):
    start1, end1 = span1
    start2, end2 = span2
    inter = max(0, min(end1, end2) - max(start1, start2))
    union = max(end1, end2) - min(start1, start2)
    return inter / union if union > 0 else 0

def normalize_sentiment(sentiment):
    sentiment = sentiment.strip().lower()
    if sentiment in {'limitation', 'suggestion'}:
        return 'Negative'
    return sentiment.capitalize()

def merge_spans_majority_aspect(annotations, iou_threshold=0.5, min_votes=2):
    """
    annotations: list of list of [start, end, label]
    returns: list of dicts with majority-agreed aspect-sentiment pairs per span
    """
    all_labeled_spans = []

    for ann_list in annotations:
        for span in ann_list:
            all_labeled_spans.append((span[0], span[1], span[2]))

    # Group spans using IoU
    span_clusters = []
    for span in all_labeled_spans:
        matched = False
        for cluster in span_clusters:
            if any(iou(span[:2], s[:2]) >= iou_threshold for s in cluster):
                cluster.append(span)
                matched = True
                break
        if not matched:
            span_clusters.append([span])

    merged = []
    for cluster in span_clusters:
        if len(cluster) >= min_votes:
            starts = [s[0] for s in cluster]
            ends = [s[1] for s in cluster]
            raw_labels = [s[2] for s in cluster]
            
            pair_counter = Counter()
            for label in raw_labels:
                if ' - ' in label:
                    aspect, sentiment = label.split(' - ', maxsplit=1)
                    aspect = aspect.strip()
                    norm_sentiment = normalize_sentiment(sentiment)
                    pair_counter[(aspect, norm_sentiment)] += 1

            final_pairs = [(a, s) for (a, s), count in pair_counter.items() if count >= min_votes]

            aspects = [a for a, s in final_pairs]
            sentiments = [s for a, s in final_pairs]

            merged.append({
                'start': min(starts),
                'end': max(ends),
                'aspects': aspects,
                'sentiments': sentiments,
                'raw_labels': raw_labels
            })

    return merged

def merge_all_annotations(df1, df2, df3, iou_threshold):
    result_rows = []

    for i in range(len(df1)):
        ann1 = df1.loc[i, 'entities']
        ann2 = df2.loc[i, 'entities']
        ann3 = df3.loc[i, 'entities']
        
        merged = merge_spans_majority_aspect([ann1, ann2, ann3], iou_threshold)
        
        result_rows.append({
            'text': df1.loc[i, 'text'],
            'merged_entities': merged
        })

    return pd.DataFrame(result_rows)

def is_valid_row(merged_entities):
    if not isinstance(merged_entities, list):
        return False
    if not merged_entities:
        return False
    return any(entity.get('aspects') for entity in merged_entities)

def convert_to_annotated_format(row):
    result = []

    for span in row['entities']:
        start = span[0]
        end = span[1]
        label = span[2]
        if ' - ' in label:
            aspect, sentiment = label.split(' - ', maxsplit=1)
            aspect = aspect.strip()
            norm_sentiment = normalize_sentiment(sentiment)
        
        result.append({
            'start': start,
            'end': end,
            'aspects': [aspect],
            'sentiments': [norm_sentiment],
            'raw_label': label
        })

    return result

In [31]:
result_df = merge_all_annotations(res_1, res_2, res_3, iou_threshold=0.6)
disagreements = result_df[~result_df['merged_entities'].apply(is_valid_row)]
dis_idx = disagreements.index.tolist()

res_4 = pd.read_json('datasets/annotator1_revised.jsonl', lines=True)
filtered = res_4.loc[dis_idx]
revised_annotations = filtered.apply(convert_to_annotated_format, axis=1)
result_df.loc[dis_idx, 'merged_entities'] = revised_annotations.values
result_df.to_csv('datasets/majority_voting_results.csv')
disagreements = result_df[~result_df['merged_entities'].apply(is_valid_row)]
print(f"Disagreements after revision: {len(disagreements)}")

In [12]:
# df = pd.read_csv('datasets/majority_voting_results.csv')
# df['merged_entities'] = df['merged_entities'].apply(ast.literal_eval)
# df = df[df['merged_entities'].apply(lambda entities: all('Neutral' not in (entity.get('sentiments', []) if isinstance(entity, dict) else []) for entity in entities))]

# df.to_csv('datasets/majority_voting_results_filtered.csv', index=False)
